# Fine-tune FunctionGemma for banking tool routing

[Open in Colab](https://colab.research.google.com/github/LxYuan0420/nlp/blob/main/notebooks/Finetune_FunctionGemma_Banking77_Tool_Router_with_TRL_Colab.ipynb)

This is an executable, top-to-bottom tutorial. Each code cell performs one visible stage: prepare BANKING77, inspect FunctionGemma's native tool format, configure TRL, measure the base model, train, evaluate, save, publish, and try new messages.

The reusable implementation remains in the repository's `uv` script. This notebook calls its public components one stage at a time instead of hiding the experiment behind one script execution.

### Verified reference run

A free T4 trained the 270M-parameter model for 400 optimizer steps in 18 minutes 22 seconds. Exact first-tool accuracy improved from **51% to 97%** over 100 deterministic held-out requests. Validation loss was lowest at epoch 2 and rose slightly by epoch 4, demonstrating mild overfitting.

## 1. Prerequisites

1. Choose **Runtime → Change runtime type → T4 GPU**. Free GPU allocation is best effort.
2. Accept the [FunctionGemma license](https://huggingface.co/google/functiongemma-270m-it).
3. Create a write-capable [Hugging Face token](https://huggingface.co/settings/tokens).
4. Add it through Colab's key icon as a secret named `HF_TOKEN` and enable notebook access.

The token controls the Hugging Face namespace. The Google account used for Colab does not determine the Hub username.

In [ ]:
import os

import torch
from google.colab import userdata

if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a write-capable HF_TOKEN in Colab Secrets.")
os.environ["HF_TOKEN"] = hf_token

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")
print("Hugging Face token loaded (value hidden).")

## 2. Install the verified stack

These are the versions used by the reference experiment. TensorBoard provides live charts inside Colab. Trackio records the same training run locally and later publishes a free static dashboard on Hugging Face.

In [ ]:
%pip install -q accelerate==1.14.0 datasets==5.0.1 'huggingface-hub>=1.4.0,<2' sentencepiece 'tensorboard==2.20.0' trackio==0.37.0 transformers==5.16.1 trl==1.12.0

## 3. Load the reusable implementation

The notebook imports the repository script as a module. We will call its dataset preparer, evaluator, experiment runner, and publisher in separate cells. This keeps one implementation while making every stage observable.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

repo_dir = Path("/content/nlp")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/LxYuan0420/nlp.git", str(repo_dir)],
        check=True,
    )

script_path = repo_dir / "scripts" / "finetune_functiongemma_banking77_colab.py"
module_name = "functiongemma_banking77_experiment"
spec = importlib.util.spec_from_file_location(module_name, script_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {script_path}")
experiment = importlib.util.module_from_spec(spec)
sys.modules[module_name] = experiment
spec.loader.exec_module(experiment)
print(f"Loaded: {script_path}")

## 4. Define the experiment

We use ten of BANKING77's 77 intents, with 80 training and 20 held-out examples per intent. Four epochs are intentionally long enough to expose when validation loss starts separating from training loss.

The effective batch is 8: four examples per device multiplied by two gradient-accumulation steps.

In [ ]:
from dataclasses import asdict
from IPython.display import JSON, Markdown, display
from transformers import set_seed

config = experiment.ExperimentConfig(
    output_dir="/content/functiongemma-banking77-router",
    epochs=4.0,
    train_examples_per_intent=80,
    eval_examples_per_intent=20,
    generation_eval_limit=100,
    seed=42,
    publish=True,
)
set_seed(config.seed)
runner = experiment.FunctionGemmaExperiment(config)
evaluator = experiment.ToolRoutingEvaluator()
runner.output_dir.mkdir(parents=True, exist_ok=True)
display(JSON(asdict(config)))

## 5. Prepare the dataset

BANKING77 begins as classification data:

```json
{
  "text": "My card is lost",
  "label_text": "lost_or_stolen_card"
}
```

We do not attach a classification head. The preparer converts `label_text` into a target assistant function call, while `text` becomes both the user message and the tool argument. The returned dataset has `messages` and `tools` columns.

In [ ]:
preparer = experiment.BankingToolDatasetPreparer(config)
dataset = preparer.prepare()
sample = dataset["train"][0]
expected_tool = evaluator.expected_tool(sample)

print(f"Train rows: {len(dataset['train'])}")
print(f"Held-out rows: {len(dataset['test'])}")
print(f"Columns: {dataset['train'].column_names}")
display(Markdown("### Classification-like source view"))
display(JSON({
    "text": sample["messages"][1]["content"],
    "label_text": expected_tool.removeprefix("handle_"),
}))
display(Markdown("### Transformed target messages"))
display(JSON(sample["messages"]))
display(Markdown("### One complete tool schema"))
display(JSON(sample["tools"][0]))

### What changed?

The original class remains conceptually present, but it is now represented by a generated function name such as `handle_card_arrival`. FunctionGemma also sees all ten available schemas, so it learns the application contract as well as the intent mapping.

This makes evaluation stricter than ordinary classification: the model must generate a complete FunctionGemma call and select the exact expected function name.

## 6. Load the model

`google/functiongemma-270m-it` is small enough for a full fine-tune on a T4. We load FP32 master weights and let Trainer use FP16 autocast during training. The saved result is a standalone model rather than a LoRA adapter.

In [ ]:
model, tokenizer = runner.load_model()
runner.summarize_model(
    model,
    tokenizer,
    sample,
    show_rendered_prompt=False,
)

## 7. Inspect what the model actually receives

The dataset stores structured Python dictionaries, but the model receives tokens. FunctionGemma's chat template serializes the developer instruction, ten schemas, user text, and expected assistant call using its control tokens.

This cell deliberately prints one full rendered training example. It is verbose because it exposes the real model input rather than a simplified proxy.

In [ ]:
rendered_prompt = tokenizer.apply_chat_template(
    sample["messages"],
    tools=sample["tools"],
    add_generation_prompt=False,
    tokenize=False,
)
rendered_token_ids = tokenizer(rendered_prompt, add_special_tokens=False)["input_ids"]
print(f"Rendered token count: {len(rendered_token_ids)}")
print(rendered_prompt)

## 8. Prove targets are not truncated

The function call is at the end of each rendered row. If a row exceeds the configured 1,024-token context, training could silently remove part of the answer. We therefore render and count every selected example before training.

In [ ]:
runner.validate_prompt_lengths(dataset, tokenizer)

## 9. Start TensorBoard before training

TensorBoard reads event files while Trainer writes them, so start it before the training cell. Open the Scalars view to watch training loss, evaluation loss, learning rate, gradient norm, and token accuracy.

In [ ]:
tensorboard_dir = str(runner.tensorboard_dir)
%load_ext tensorboard
%tensorboard --logdir $tensorboard_dir

## 10. Configure TRL and understand the loss

TRL 1.12 calls the default loss `chunked_nll`. Mathematically it is normal next-token negative log-likelihood, or cross-entropy:

```text
loss = mean(-log P(correct next token | previous tokens))
```

Standard `nll` may materialize logits shaped `[batch, sequence, vocabulary]`. `chunked_nll` projects hidden states into vocabulary logits in smaller chunks and accumulates the same loss, reducing peak memory. It does **not** change which tokens contribute.

Here, `assistant_only_loss=False` and `completion_only_loss=False`. Consequently, every non-padding token in the developer prompt, schemas, user message, and assistant call contributes. Padding positions are marked `-100` and ignored. See the [TRL 1.12 SFT documentation](https://huggingface.co/docs/trl/v1.12.0/en/sft_trainer).

In [ ]:
trainer = runner.build_trainer(model, tokenizer, dataset)

print(f"Loss type: {trainer.args.loss_type}")
print(f"Assistant-only loss: {trainer.args.assistant_only_loss}")
print(f"Completion-only loss: {trainer.args.completion_only_loss}")
print(f"FP16 autocast: {trainer.args.fp16}")

## 11. Measure the untouched model

Before training, generate calls for the deterministic held-out subset. The task metric extracts the first complete function call and requires an exact function-name match. This baseline prevents us from mistaking an already-capable base model for a fine-tuning improvement.

In [ ]:
baseline_accuracy, baseline_predictions = evaluator.evaluate(
    trainer.model,
    tokenizer,
    dataset["test"],
    config.generation_eval_limit,
)
evaluator.print_results(
    "BEFORE FINE-TUNING",
    baseline_accuracy,
    baseline_predictions,
)

## 12. Train

This is the expensive cell. The default run performs four epochs, evaluates and saves once per epoch, and logs every ten optimizer steps. On the verified free T4 it took about 18 minutes for training, excluding setup and generation evaluation.

In [ ]:
train_result = trainer.train()
train_result.metrics

## 13. Evaluate the fine-tuned model

Trainer evaluation reports token-level loss and token accuracy. We separately repeat deterministic generation to measure the behavior we actually want: choosing the correct banking tool.

In [ ]:
eval_metrics = trainer.evaluate()
final_accuracy, final_predictions = evaluator.evaluate(
    trainer.model,
    tokenizer,
    dataset["test"],
    config.generation_eval_limit,
)

print(f"Baseline exact tool accuracy: {baseline_accuracy:.2%}")
print(f"Final exact tool accuracy:    {final_accuracy:.2%}")
print(f"Final validation loss:        {eval_metrics['eval_loss']:.4f}")
evaluator.print_results("AFTER FINE-TUNING", final_accuracy, final_predictions)

## 14. Inspect epoch-level learning behavior

A falling training loss with a rising validation loss means the model is becoming more confident on seen examples without generalizing equally to unseen examples. Cross-entropy can rise because of a few confidently wrong tokens even when token accuracy barely changes.

In [ ]:
epoch_evaluations = [
    {
        "epoch": row["epoch"],
        "eval_loss": row["eval_loss"],
        "eval_token_accuracy": row.get("eval_mean_token_accuracy"),
    }
    for row in trainer.state.log_history
    if "eval_loss" in row
]
display(JSON(epoch_evaluations))

## 15. Save the model and experiment evidence

The save stage writes standalone model weights, tokenizer files, Trainer state, TensorBoard events, exact configuration, package versions, and all before/after predictions. `training_metrics.json` becomes the source of truth for the generated model card.

In [ ]:
metrics_path = runner.save_artifacts(
    trainer=trainer,
    tokenizer=tokenizer,
    train_metrics=train_result.metrics,
    eval_metrics=eval_metrics,
    baseline_accuracy=baseline_accuracy,
    final_accuracy=final_accuracy,
    baseline_predictions=baseline_predictions,
    final_predictions=final_predictions,
)
print(metrics_path)

## 16. Generate the model card and publish

Publication generates `README.md` directly from the recorded metrics, syncs Trackio to a free static Space, uploads the complete model folder while excluding intermediate checkpoints, and verifies the required remote files. No separate model-card script is involved.

In [ ]:
publication = runner.publish(metrics_path)
if publication is None:
    print("Publication disabled; artifacts remain local.")
else:
    display(Markdown(
        f"- [Published model]({publication.model_url})\n"
        f"- [Trackio dashboard]({publication.trackio_url})"
    ))

## 17. Try new customer messages

The model returns a proposed function call; it does not execute banking code. A real application must parse the output, validate arguments, and dispatch through an explicit allow-listed mapping.

These examples reuse the trained model already in GPU memory and demonstrate that different inputs select different functions.

In [ ]:
@torch.inference_mode()
def route_customer_message(customer_message: str) -> tuple[str, str]:
    messages = [
        {"role": "developer", "content": experiment.DEVELOPER_PROMPT},
        {"role": "user", "content": customer_message},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tools=experiment.TOOLS,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(trainer.model.device)
    output = trainer.model.generate(
        **inputs,
        max_new_tokens=experiment.MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    generated = tokenizer.decode(
        output[0, inputs["input_ids"].shape[1] :],
        skip_special_tokens=False,
    )
    return evaluator.first_function_call(generated)

examples = [
    "My card was stolen last night",
    "Why has my card payment not completed?",
    "How do I change my PIN?",
    "Please close my account",
]
for customer_message in examples:
    selected_tool, raw_call = route_customer_message(customer_message)
    print(f"Input: {customer_message}")
    print(f"Tool:  {selected_tool}")
    print(f"Call:  {raw_call}\n")

## What to try next

- Evaluate exact generated tool accuracy after every epoch and select the best checkpoint by that task metric.
- Compare full-sequence loss with a carefully verified completion-only formulation. FunctionGemma's current chat template does not expose TRL assistant-generation masks, so `assistant_only_loss=True` is not a safe one-line change.
- Expand from ten to all 77 intents and examine confusion among similar routes.
- Add an out-of-scope or refusal route so unrelated messages are not forced into a banking handler.
- Compare full fine-tuning with LoRA using the same seed, split, and generated evaluation.
- Score argument fidelity separately from function-name selection.